# A space is something. An empty string is nothing.

**Python Foundation · An interactive experiment**

Choose **Runtime → Run all** in Google Colab. Then edit the text and the search string in the explorer. All code cells are editable and run real Python.

Two expressions can both return `True` for different reasons:

```python
x = "hello world"
print(' ' in x)  # True: x contains an actual space.
print('' in x)   # True: the empty string is a substring of every string.
```

`' '` contains one space character. `''` contains zero characters. The quotation marks delimit the string; they are not part of its contents.

In [ ]:
x = "hello world"  # Try "hello", "", or "   ".
for needle in [" ", ""]:
    print(f"Search {needle!r}: length={len(needle)}, contained={needle in x}")
print(f"Text: {x!r}, length={len(x)}")

## Why does nothing match?

A substring matches when a slice of the text equals the search string. For an empty search string, a zero-length slice at any boundary is empty:

```python
x[i:i] == ''
```

A string of length **n** has **n + 1 boundaries**: before the first character, between characters, and after the last character. Even `''` has one boundary. This is a useful way to visualize Python's empty-substring rule; no actual space character is required.

`in` asks whether a match exists. `find` gives the first match's index (or `-1`). `count` counts non-overlapping matches; specifically, `x.count('')` is `len(x) + 1`.

In [ ]:
for text in ["hello world", "hello", "", "   "]:
    print(f"{text!r:15} | space: {' ' in text!s:5} | empty: {'' in text!s:5} | empty count: {text.count('')}")

## Explore the matches

Use **Space** and **Empty** to compare the two cases. You can also type a custom search string. The diagram shows all matching start positions, including overlapping matches. The `count` result below uses Python's non-overlapping counting rule.

In the diagram, **␠** represents an actual space. Boundary markers represent positions, not characters.

In [ ]:
import html
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

def matching_positions(text, needle):
    return [i for i in range(len(text) - len(needle) + 1)
            if text[i:i + len(needle)] == needle]

def render_experiment(text, needle):
    positions = matching_positions(text, needle)
    visible = lambda c: "␠" if c == " " else "↵" if c == "\n" else "⇥" if c == "\t" else html.escape(c)
    # Keep the visual readable for long inputs; all computed results remain exact.
    limit = min(len(text), 60)
    pieces = []
    for i in range(limit + 1):
        hit = i in positions
        color = "#167a96" if hit else "#64748b"
        pieces.append(f'<span style="display:inline-flex;flex-direction:column;align-items:center;min-width:20px;color:{color}"><b>{"●" if hit else "│"}</b><small>{i}</small></span>')
        if i < limit:
            highlighted = any(start <= i < start + len(needle) for start in positions)
            bg = "#d9f0f6" if highlighted else "#eef2f6"
            pieces.append(f'<span style="padding:10px 12px;border-radius:7px;background:{bg};color:#172b3a">{visible(text[i])}</span>')
    detail = "Empty search: every boundary matches, without consuming a character." if needle == "" else "A space is one real character; it matches only where that character occurs." if needle == " " else "Matching slices must contain exactly the characters you searched for."
    result = f"{needle!r} in {text!r} → {needle in text}"
    position_label = str(positions[:61]) + (" …" if len(positions) > 61 else "")
    return f"""<div style="padding:22px;border:1px solid #cbd5e1;border-radius:14px;background:#fff;color:#172b3a;font:15px/1.7 system-ui">
    <strong style="font-size:20px">{html.escape(result)}</strong>
    <p>Search length: <b>{len(needle)}</b> · Text length: <b>{len(text)}</b> · First match: <b>{text.find(needle)}</b> · Python count: <b>{text.count(needle)}</b></p>
    <div style="display:flex;align-items:center;flex-wrap:wrap;gap:3px;font-family:monospace">{''.join(pieces)}</div>
    {'<p>Diagram shows the first 60 characters.</p>' if len(text) > limit else ''}
    <p>Matching start positions: {html.escape(position_label)}</p><p>{detail}</p></div>"""

text_input = widgets.Text(value="hello world", description="Text:", layout=widgets.Layout(width="95%"))
needle_input = widgets.Text(value=" ", description="Search:", placeholder="Leave blank for the empty string", layout=widgets.Layout(width="95%"))
space_button = widgets.Button(description="Space: ' '")
empty_button = widgets.Button(description="Empty: ''")
reset_button = widgets.Button(description="Reset")
view = widgets.HTML()

def update(change=None):
    view.value = render_experiment(text_input.value, needle_input.value)

def use_space(button):
    needle_input.value = " "
def use_empty(button):
    needle_input.value = ""
def reset(button):
    text_input.value = "hello world"
    needle_input.value = " "

text_input.observe(update, names="value")
needle_input.observe(update, names="value")
space_button.on_click(use_space)
empty_button.on_click(use_empty)
reset_button.on_click(reset)
update()
display(widgets.VBox([text_input, needle_input, widgets.HBox([space_button, empty_button, reset_button]), view]))

## Predict, then run

1. If `x = "hello"`, will `' ' in x` still be `True`?
2. If `x = ""`, what are the results of both membership checks?
3. How many boundaries does `"abc"` have?

Change the code below and check your predictions.

In [ ]:
x = "hello"
print("Contains a space:", " " in x)
print("Contains the empty string:", "" in x)
print("Empty-substring matches:", x.count(""))
print("Zero-length slices:", [x[i:i] for i in range(len(x) + 1)])

## Remember

| Search string | Length | When does `needle in x` return `True`? |
|---|---:|---|
| `' '` | 1 | When `x` contains a space character. |
| `''` | 0 | Always, for every string `x`, including `''`. |

A space is not an empty string. **Equal Boolean results do not imply equal reasons.**

Further reading: [Python string operations](https://docs.python.org/3/library/stdtypes.html#string-methods).